In [9]:
"""
Complete India AQI + Weather Data Extraction Pipeline
======================================================
All 28 States + 8 Union Territories Coverage
Extracts all required features for AQI prediction model

Output Format:
City, datetime, PM2.5, PM10, NO, NO2, NOx, NH3, SO2, CO, O3,
Temperature, Humidity, Pressure, Wind_Speed, Year, Month, Day, Hour,
DayOfWeek, Season, Is_Weekend, Is_Rush_Hour, PM2.5_Lag_1h, PM2.5_Lag_24h,
PM2.5_Rolling_Mean_24h, AQI
"""

import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time
import json
import os
from typing import Dict, List, Optional

# ============================================
# CONFIGURATION
# ============================================

WAQI_TOKEN = os.environ.get("WAQI_TOKEN", "")

# All 28 States + 8 Union Territories (36 locations total)
INDIA_CITIES = {
    # ========== 28 STATES ==========
    # Andhra Pradesh
    "Visakhapatnam": {
        "lat": 17.6868, "lon": 83.2185, 
        "state": "Andhra Pradesh", "search": "visakhapatnam"
    },
    # Arunachal Pradesh
    "Itanagar": {
        "lat": 27.0844, "lon": 93.6053,
        "state": "Arunachal Pradesh", "search": "itanagar"
    },
    # Assam
    "Guwahati": {
        "lat": 26.1445, "lon": 91.7362,
        "state": "Assam", "search": "guwahati"
    },
    # Bihar
    "Patna": {
        "lat": 25.5941, "lon": 85.1376,
        "state": "Bihar", "search": "patna"
    },
    # Chhattisgarh
    "Raipur": {
        "lat": 21.2514, "lon": 81.6296,
        "state": "Chhattisgarh", "search": "raipur"
    },
    # Goa
    "Panaji": {
        "lat": 15.4909, "lon": 73.8278,
        "state": "Goa", "search": "goa"
    },
    # Gujarat
    "Ahmedabad": {
        "lat": 23.0225, "lon": 72.5714,
        "state": "Gujarat", "search": "ahmedabad"
    },
    # Haryana
    "Faridabad": {
        "lat": 28.4089, "lon": 77.3178,
        "state": "Haryana", "search": "faridabad"
    },
    # Himachal Pradesh
    "Shimla": {
        "lat": 31.1048, "lon": 77.1734,
        "state": "Himachal Pradesh", "search": "shimla"
    },
    # Jharkhand
    "Ranchi": {
        "lat": 23.3441, "lon": 85.3096,
        "state": "Jharkhand", "search": "ranchi"
    },
    # Karnataka
    "Bengaluru": {
        "lat": 12.9716, "lon": 77.5946,
        "state": "Karnataka", "search": "bangalore"
    },
    # Kerala
    "Thiruvananthapuram": {
        "lat": 8.5241, "lon": 76.9366,
        "state": "Kerala", "search": "thiruvananthapuram"
    },
    # Madhya Pradesh
    "Indore": {
        "lat": 22.7196, "lon": 75.8577,
        "state": "Madhya Pradesh", "search": "indore"
    },
    # Maharashtra
    "Mumbai": {
        "lat": 19.0760, "lon": 72.8777,
        "state": "Maharashtra", "search": "mumbai"
    },
    # Manipur
    "Imphal": {
        "lat": 24.8170, "lon": 93.9368,
        "state": "Manipur", "search": "imphal"
    },
    # Meghalaya
    "Shillong": {
        "lat": 25.5788, "lon": 91.8933,
        "state": "Meghalaya", "search": "shillong"
    },
    # Mizoram
    "Aizawl": {
        "lat": 23.7307, "lon": 92.7173,
        "state": "Mizoram", "search": "aizawl"
    },
    # Nagaland
    "Kohima": {
        "lat": 25.6747, "lon": 94.1086,
        "state": "Nagaland", "search": "kohima"
    },
    # Odisha
    "Bhubaneswar": {
        "lat": 20.2961, "lon": 85.8245,
        "state": "Odisha", "search": "bhubaneswar"
    },
    # Punjab
    "Ludhiana": {
        "lat": 30.9010, "lon": 75.8573,
        "state": "Punjab", "search": "ludhiana"
    },
    # Rajasthan
    "Jaipur": {
        "lat": 26.9124, "lon": 75.7873,
        "state": "Rajasthan", "search": "jaipur"
    },
    # Sikkim
    "Gangtok": {
        "lat": 27.3389, "lon": 88.6065,
        "state": "Sikkim", "search": "gangtok"
    },
    # Tamil Nadu
    "Chennai": {
        "lat": 13.0827, "lon": 80.2707,
        "state": "Tamil Nadu", "search": "chennai"
    },
    # Telangana
    "Hyderabad": {
        "lat": 17.3850, "lon": 78.4867,
        "state": "Telangana", "search": "hyderabad"
    },
    # Tripura
    "Agartala": {
        "lat": 23.8315, "lon": 91.2868,
        "state": "Tripura", "search": "agartala"
    },
    # Uttar Pradesh
    "Lucknow": {
        "lat": 26.8467, "lon": 80.9462,
        "state": "Uttar Pradesh", "search": "lucknow"
    },
    # Uttarakhand
    "Dehradun": {
        "lat": 30.3165, "lon": 78.0322,
        "state": "Uttarakhand", "search": "dehradun"
    },
    # West Bengal
    "Kolkata": {
        "lat": 22.5726, "lon": 88.3639,
        "state": "West Bengal", "search": "kolkata"
    },
    
    # ========== 8 UNION TERRITORIES ==========
    # Delhi (National Capital Territory)
    "Delhi": {
        "lat": 28.6139, "lon": 77.2090,
        "state": "Delhi (NCT)", "search": "delhi"
    },
    # Chandigarh
    "Chandigarh": {
        "lat": 30.7333, "lon": 76.7794,
        "state": "Chandigarh (UT)", "search": "chandigarh"
    },
    # Puducherry
    "Puducherry": {
        "lat": 11.9416, "lon": 79.8083,
        "state": "Puducherry (UT)", "search": "puducherry"
    },
    # Jammu & Kashmir
    "Srinagar": {
        "lat": 34.0837, "lon": 74.7973,
        "state": "Jammu & Kashmir (UT)", "search": "srinagar"
    },
    # Ladakh
    "Leh": {
        "lat": 34.1526, "lon": 77.5771,
        "state": "Ladakh (UT)", "search": "leh"
    },
    # Andaman & Nicobar Islands
    "Port_Blair": {
        "lat": 11.6234, "lon": 92.7265,
        "state": "Andaman & Nicobar (UT)", "search": "port blair"
    },
    # Dadra & Nagar Haveli and Daman & Diu
    "Daman": {
        "lat": 20.4283, "lon": 72.8397,
        "state": "Daman & Diu (UT)", "search": "daman"
    },
    # Lakshadweep
    "Kavaratti": {
        "lat": 10.5667, "lon": 72.6417,
        "state": "Lakshadweep (UT)", "search": "kavaratti"
    }
}

# Indian Holidays (2015-2024)
INDIAN_HOLIDAYS = [
    # Republic Day
    *[f"{year}-01-26" for year in range(2015, 2025)],
    # Independence Day
    *[f"{year}-08-15" for year in range(2015, 2025)],
    # Gandhi Jayanti
    *[f"{year}-10-02" for year in range(2015, 2025)],
    # Diwali (approximate dates)
    "2015-11-11", "2016-10-30", "2017-10-19", "2018-11-07",
    "2019-10-27", "2020-11-14", "2021-11-04", "2022-10-24",
    "2023-11-12", "2024-11-01",
    # Holi
    "2015-03-06", "2016-03-24", "2017-03-13", "2018-03-02",
    "2019-03-21", "2020-03-10", "2021-03-29", "2022-03-18",
    "2023-03-08", "2024-03-25",
]


# ============================================
# WEATHER DATA FETCHER (Open-Meteo)
# ============================================

class WeatherDataFetcher:
    """
    Fetch historical and current weather data using Open-Meteo API
    Free, no API key required
    """
    
    def __init__(self):
        self.historical_url = "https://archive-api.open-meteo.com/v1/archive"
        self.current_url = "https://api.open-meteo.com/v1/forecast"
    
    def fetch_historical_weather(self, city: str, lat: float, lon: float, 
                                  start_date: str, end_date: str) -> pd.DataFrame:
        """
        Fetch historical hourly weather data
        
        Parameters:
        -----------
        city : str
            City name
        lat, lon : float
            Coordinates
        start_date, end_date : str
            Format: 'YYYY-MM-DD'
        """
        
        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": start_date,
            "end_date": end_date,
            "hourly": [
                "temperature_2m",
                "relative_humidity_2m",
                "surface_pressure",
                "wind_speed_10m",
                "wind_direction_10m",
                "precipitation",
                "cloud_cover",
                "visibility"
            ],
            "timezone": "Asia/Kolkata"
        }
        
        try:
            response = requests.get(self.historical_url, params=params, timeout=60)
            data = response.json()
            
            if 'hourly' in data:
                hourly = data['hourly']
                df = pd.DataFrame({
                    'City': city,
                    'datetime': pd.to_datetime(hourly['time']),
                    'Temperature': hourly.get('temperature_2m'),
                    'Humidity': hourly.get('relative_humidity_2m'),
                    'Pressure': hourly.get('surface_pressure'),
                    'Wind_Speed': hourly.get('wind_speed_10m'),
                    'Wind_Direction': hourly.get('wind_direction_10m'),
                    'Precipitation': hourly.get('precipitation'),
                    'Cloud_Cover': hourly.get('cloud_cover'),
                    'Visibility': hourly.get('visibility')
                })
                return df
            
        except Exception as e:
            print(f"    Error fetching weather for {city}: {e}")
        
        return pd.DataFrame()
    
    def fetch_current_weather(self, city: str, lat: float, lon: float) -> dict:
        """
        Fetch current weather data
        """
        
        params = {
            "latitude": lat,
            "longitude": lon,
            "current": [
                "temperature_2m",
                "relative_humidity_2m",
                "surface_pressure",
                "wind_speed_10m",
                "wind_direction_10m",
                "precipitation",
                "cloud_cover"
            ],
            "timezone": "Asia/Kolkata"
        }
        
        try:
            response = requests.get(self.current_url, params=params, timeout=30)
            data = response.json()
            
            if 'current' in data:
                current = data['current']
                return {
                    'City': city,
                    'datetime': datetime.now(),
                    'Temperature': current.get('temperature_2m'),
                    'Humidity': current.get('relative_humidity_2m'),
                    'Pressure': current.get('surface_pressure'),
                    'Wind_Speed': current.get('wind_speed_10m'),
                    'Wind_Direction': current.get('wind_direction_10m'),
                    'Precipitation': current.get('precipitation'),
                    'Cloud_Cover': current.get('cloud_cover')
                }
        
        except Exception as e:
            print(f"    Error: {e}")
        
        return {}


# ============================================
# AQI DATA FETCHER (WAQI + OpenAQ)
# ============================================

class AQIDataFetcher:
    """
    Fetch AQI data from WAQI and OpenAQ
    """
    
    def __init__(self, waqi_token: str):
        self.waqi_token = waqi_token
        self.waqi_url = "https://api.waqi.info"
        self.openaq_url = "https://api.openaq.org/v2"
    
    def fetch_waqi_current(self, city: str, search_term: str) -> dict:
        """
        Fetch current AQI data from WAQI
        """
        
        url = f"{self.waqi_url}/feed/{search_term}/?token={self.waqi_token}"
        
        try:
            response = requests.get(url, timeout=30)
            data = response.json()
            
            if data.get('status') == 'ok':
                aqi_data = data['data']
                iaqi = aqi_data.get('iaqi', {})
                time_info = aqi_data.get('time', {})
                
                return {
                    'City': city,
                    'datetime': time_info.get('s'),
                    'PM2.5': iaqi.get('pm25', {}).get('v'),
                    'PM10': iaqi.get('pm10', {}).get('v'),
                    'NO': iaqi.get('no', {}).get('v'),
                    'NO2': iaqi.get('no2', {}).get('v'),
                    'NH3': iaqi.get('nh3', {}).get('v'),
                    'SO2': iaqi.get('so2', {}).get('v'),
                    'CO': iaqi.get('co', {}).get('v'),
                    'O3': iaqi.get('o3', {}).get('v'),
                    'Temperature': iaqi.get('t', {}).get('v'),
                    'Humidity': iaqi.get('h', {}).get('v'),
                    'Pressure': iaqi.get('p', {}).get('v'),
                    'Wind_Speed': iaqi.get('w', {}).get('v'),
                    'AQI': aqi_data.get('aqi'),
                    'Dominant_Pollutant': aqi_data.get('dominentpol'),
                    'Source': 'WAQI'
                }
        
        except Exception as e:
            print(f"    WAQI Error for {city}: {e}")
        
        return {}
    
    def fetch_openaq_historical(self, city: str, days: int = 365) -> pd.DataFrame:
        """
        Fetch historical AQI data from OpenAQ
        """
        
        date_from = (datetime.now() - timedelta(days=days)).strftime("%Y-%m-%d")
        
        parameters = ['pm25', 'pm10', 'no', 'no2', 'so2', 'co', 'o3', 'nh3']
        all_data = []
        
        for param in parameters:
            url = f"{self.openaq_url}/measurements"
            params = {
                "city": city,
                "country": "IN",
                "parameter": param,
                "date_from": date_from,
                "limit": 10000
            }
            
            try:
                response = requests.get(url, params=params, timeout=60)
                data = response.json()
                
                for m in data.get('results', []):
                    all_data.append({
                        'City': city,
                        'datetime': m['date']['utc'],
                        'parameter': param.upper(),
                        'value': m['value']
                    })
                
            except Exception as e:
                pass
        
        if all_data:
            df = pd.DataFrame(all_data)
            df['datetime'] = pd.to_datetime(df['datetime'])
            
            # Pivot to get parameters as columns
            df_pivot = df.pivot_table(
                index=['City', 'datetime'],
                columns='parameter',
                values='value',
                aggfunc='mean'
            ).reset_index()
            
            # Rename columns
            df_pivot.columns.name = None
            column_mapping = {
                'PM25': 'PM2.5',
                'PM10': 'PM10',
                'NO': 'NO',
                'NO2': 'NO2',
                'NH3': 'NH3',
                'SO2': 'SO2',
                'CO': 'CO',
                'O3': 'O3'
            }
            df_pivot.rename(columns=column_mapping, inplace=True)
            
            return df_pivot
        
        return pd.DataFrame()


# ============================================
# FEATURE ENGINEERING
# ============================================

class FeatureEngineer:
    """
    Create all required features for AQI prediction
    """
    
    def __init__(self):
        self.holidays = INDIAN_HOLIDAYS
    
    def add_temporal_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Add all temporal features
        """
        
        df = df.copy()
        df['datetime'] = pd.to_datetime(df['datetime'])
        
        # Basic temporal features
        df['Year'] = df['datetime'].dt.year
        df['Month'] = df['datetime'].dt.month
        df['Day'] = df['datetime'].dt.day
        df['Hour'] = df['datetime'].dt.hour
        df['DayOfWeek'] = df['datetime'].dt.dayofweek  # 0=Monday, 6=Sunday
        
        # Season (Indian seasons)
        def get_season(month):
            if month in [12, 1, 2]:
                return 1  # Winter
            elif month in [3, 4, 5]:
                return 2  # Summer
            elif month in [6, 7, 8, 9]:
                return 3  # Monsoon
            else:
                return 4  # Post-Monsoon/Autumn
        
        df['Season'] = df['Month'].apply(get_season)
        
        # Weekend indicator
        df['Is_Weekend'] = (df['DayOfWeek'] >= 5).astype(int)
        
        # Rush hour indicator (8-10 AM and 5-8 PM)
        df['Is_Rush_Hour'] = df['Hour'].apply(
            lambda h: 1 if h in [8, 9, 10, 17, 18, 19, 20] else 0
        )
        
        # Holiday indicator
        df['date_str'] = df['datetime'].dt.strftime('%Y-%m-%d')
        df['Is_Holiday'] = df['date_str'].isin(self.holidays).astype(int)
        df.drop('date_str', axis=1, inplace=True)
        
        # Night indicator
        df['Is_Night'] = df['Hour'].apply(
            lambda h: 1 if h in [22, 23, 0, 1, 2, 3, 4, 5] else 0
        )
        
        return df
    
    def add_lag_features(self, df: pd.DataFrame, 
                         target_col: str = 'PM2.5') -> pd.DataFrame:
        """
        Add lag and rolling features per city
        """
        
        df = df.copy()
        df = df.sort_values(['City', 'datetime'])
        
        if target_col not in df.columns:
            return df
        
        lag_dfs = []
        
        for city in df['City'].unique():
            city_df = df[df['City'] == city].copy()
            
            # Lag features
            city_df[f'{target_col}_Lag_1h'] = city_df[target_col].shift(1)
            city_df[f'{target_col}_Lag_3h'] = city_df[target_col].shift(3)
            city_df[f'{target_col}_Lag_6h'] = city_df[target_col].shift(6)
            city_df[f'{target_col}_Lag_12h'] = city_df[target_col].shift(12)
            city_df[f'{target_col}_Lag_24h'] = city_df[target_col].shift(24)
            
            # Rolling features
            city_df[f'{target_col}_Rolling_Mean_6h'] = city_df[target_col].rolling(
                window=6, min_periods=1
            ).mean()
            city_df[f'{target_col}_Rolling_Mean_24h'] = city_df[target_col].rolling(
                window=24, min_periods=1
            ).mean()
            city_df[f'{target_col}_Rolling_Std_24h'] = city_df[target_col].rolling(
                window=24, min_periods=1
            ).std()
            city_df[f'{target_col}_Rolling_Max_24h'] = city_df[target_col].rolling(
                window=24, min_periods=1
            ).max()
            city_df[f'{target_col}_Rolling_Min_24h'] = city_df[target_col].rolling(
                window=24, min_periods=1
            ).min()
            
            # Difference features
            city_df[f'{target_col}_Diff_1h'] = city_df[target_col].diff(1)
            city_df[f'{target_col}_Diff_24h'] = city_df[target_col].diff(24)
            
            lag_dfs.append(city_df)
        
        return pd.concat(lag_dfs, ignore_index=True)
    
    def add_nox_feature(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Calculate NOx from NO and NO2
        """
        
        df = df.copy()
        
        if 'NO' in df.columns and 'NO2' in df.columns:
            df['NOx'] = df['NO'].fillna(0) + df['NO2'].fillna(0)
            df['NOx'] = df['NOx'].replace(0, np.nan)
        else:
            df['NOx'] = np.nan
        
        return df
    
    def calculate_aqi(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Calculate AQI from PM2.5 using Indian AQI formula
        """
        
        def pm25_to_aqi_indian(pm25):
            if pd.isna(pm25) or pm25 < 0:
                return np.nan
            
            # Indian AQI breakpoints for PM2.5
            if pm25 <= 30:
                return pm25 * 50 / 30
            elif pm25 <= 60:
                return 50 + (pm25 - 30) * 50 / 30
            elif pm25 <= 90:
                return 100 + (pm25 - 60) * 100 / 30
            elif pm25 <= 120:
                return 200 + (pm25 - 90) * 100 / 30
            elif pm25 <= 250:
                return 300 + (pm25 - 120) * 100 / 130
            else:
                return 400 + (pm25 - 250) * 100 / 130
        
        df = df.copy()
        
        if 'AQI' not in df.columns or df['AQI'].isna().all():
            df['AQI'] = df['PM2.5'].apply(pm25_to_aqi_indian)
        
        return df


# ============================================
# COMPLETE DATA PIPELINE
# ============================================

class IndiaAQIDataPipeline:
    """
    Complete pipeline to collect and process India AQI data
    All 28 States + 8 Union Territories
    """
    
    def __init__(self, waqi_token: str):
        self.waqi_token = waqi_token
        self.weather_fetcher = WeatherDataFetcher()
        self.aqi_fetcher = AQIDataFetcher(waqi_token)
        self.feature_engineer = FeatureEngineer()
        self.cities = INDIA_CITIES
    
    def fetch_current_data_all_cities(self) -> pd.DataFrame:
        """
        Fetch current AQI + Weather data for all cities
        """
        
        print("="*80)
        print("  FETCHING CURRENT DATA - ALL 28 STATES + 8 UNION TERRITORIES")
        print("="*80)
        print(f"  Total Locations: {len(self.cities)}")
        print(f"  Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("="*80)
        
        all_data = []
        
        for idx, (city, info) in enumerate(self.cities.items(), 1):
            state = info.get('state', '')
            print(f"\n[{idx}/{len(self.cities)}] ðŸ“ {city} ({state})...", end=" ")
            
            # Fetch AQI from WAQI
            aqi_data = self.aqi_fetcher.fetch_waqi_current(city, info['search'])
            
            # Fetch Weather from Open-Meteo
            weather_data = self.weather_fetcher.fetch_current_weather(
                city, info['lat'], info['lon']
            )
            
            if aqi_data:
                # Merge AQI and Weather
                combined = {**aqi_data}
                combined['State'] = state
                
                # Use weather data if AQI doesn't have it
                if weather_data:
                    for key in ['Temperature', 'Humidity', 'Pressure', 'Wind_Speed']:
                        if combined.get(key) is None:
                            combined[key] = weather_data.get(key)
                
                all_data.append(combined)
                print(f"âœ… AQI: {combined.get('AQI', 'N/A')}")
            else:
                # Even if no AQI, save weather data
                if weather_data:
                    weather_data['State'] = state
                    all_data.append(weather_data)
                    print("âš ï¸  Weather only")
                else:
                    print("âŒ Failed")
            
            time.sleep(0.5)
        
        df = pd.DataFrame(all_data)
        
        print(f"\n{'='*80}")
        print(f"âœ… Collected data from {len(df)} locations")
        print(f"{'='*80}")
        
        return df
    
    def fetch_historical_data(self, start_date: str, end_date: str, 
                               cities: List[str] = None) -> pd.DataFrame:
        """
        Fetch historical data for specified period
        
        Parameters:
        -----------
        start_date, end_date : str
            Format: 'YYYY-MM-DD'
        cities : list
            List of city names (uses all if None)
        """
        
        if cities is None:
            cities = list(self.cities.keys())
        
        print("="*80)
        print("  FETCHING HISTORICAL DATA")
        print("="*80)
        print(f"  Period: {start_date} to {end_date}")
        print(f"  Cities: {len(cities)}")
        print("="*80)
        
        all_weather_data = []
        all_aqi_data = []
        
        for idx, city in enumerate(cities, 1):
            info = self.cities.get(city)
            if not info:
                continue
            
            state = info.get('state', '')
            print(f"\n[{idx}/{len(cities)}] ðŸ“ {city} ({state})...")
            
            # Fetch weather data
            print("  ðŸ“Š Fetching weather data...", end=" ")
            weather_df = self.weather_fetcher.fetch_historical_weather(
                city, info['lat'], info['lon'], start_date, end_date
            )
            if len(weather_df) > 0:
                weather_df['State'] = state
                all_weather_data.append(weather_df)
                print(f"âœ… {len(weather_df)} records")
            else:
                print("âŒ Failed")
            
            # Fetch AQI data from OpenAQ
            print("  ðŸŒ«ï¸  Fetching AQI data...", end=" ")
            aqi_df = self.aqi_fetcher.fetch_openaq_historical(city)
            if len(aqi_df) > 0:
                aqi_df['State'] = state
                all_aqi_data.append(aqi_df)
                print(f"âœ… {len(aqi_df)} records")
            else:
                print("âŒ No data available")
            
            time.sleep(1)
        
        # Combine weather data
        weather_combined = pd.DataFrame()
        if all_weather_data:
            weather_combined = pd.concat(all_weather_data, ignore_index=True)
        
        # Combine AQI data
        aqi_combined = pd.DataFrame()
        if all_aqi_data:
            aqi_combined = pd.concat(all_aqi_data, ignore_index=True)
        
        # Merge on City and datetime (hourly)
        if len(weather_combined) > 0 and len(aqi_combined) > 0:
            weather_combined['datetime_hour'] = weather_combined['datetime'].dt.floor('H')
            aqi_combined['datetime_hour'] = aqi_combined['datetime'].dt.floor('H')
            
            merged_df = pd.merge(
                aqi_combined,
                weather_combined,
                on=['City', 'datetime_hour'],
                how='outer',
                suffixes=('_aqi', '_weather')
            )
            
            # Use datetime from either source
            merged_df['datetime'] = merged_df['datetime_aqi'].fillna(
                merged_df['datetime_weather']
            )
            
            # Use State from either source
            merged_df['State'] = merged_df['State_aqi'].fillna(merged_df['State_weather'])
            
            return merged_df
        
        elif len(weather_combined) > 0:
            return weather_combined
        elif len(aqi_combined) > 0:
            return aqi_combined
        
        return pd.DataFrame()
    
    def process_data(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Process raw data - add all features
        """
        
        print("\n" + "="*80)
        print("  PROCESSING DATA - ADDING FEATURES")
        print("="*80)
        
        # Add temporal features
        print("  â° Adding temporal features...")
        df = self.feature_engineer.add_temporal_features(df)
        
        # Add NOx feature
        print("  ðŸ§ª Calculating NOx...")
        df = self.feature_engineer.add_nox_feature(df)
        
        # Calculate AQI
        print("  ðŸ“Š Calculating AQI...")
        df = self.feature_engineer.calculate_aqi(df)
        
        # Add lag features
        print("  ðŸ“ˆ Adding lag features...")
        df = self.feature_engineer.add_lag_features(df, 'PM2.5')
        
        print("  âœ… Processing complete!")
        
        return df
    
    def format_output(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Format output to match required structure
        """
        
        # Define column order
        columns = [
            'City', 'State', 'datetime',
            'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'SO2', 'CO', 'O3',
            'Temperature', 'Humidity', 'Pressure', 'Wind_Speed',
            'Year', 'Month', 'Day', 'Hour', 'DayOfWeek', 'Season',
            'Is_Weekend', 'Is_Rush_Hour',
            'PM2.5_Lag_1h', 'PM2.5_Lag_24h', 'PM2.5_Rolling_Mean_24h',
            'AQI'
        ]
        
        # Add missing columns with NaN
        for col in columns:
            if col not in df.columns:
                df[col] = np.nan
        
        # Select and order columns
        df_output = df[columns].copy()
        
        # Sort by State, City and datetime
        df_output = df_output.sort_values(['State', 'City', 'datetime'])
        
        return df_output
    
    def run_complete_pipeline(self, mode: str = 'current', 
                               start_date: str = None, 
                               end_date: str = None,
                               cities: List[str] = None) -> pd.DataFrame:
        """
        Run complete data collection and processing pipeline
        
        Parameters:
        -----------
        mode : str
            'current' - Fetch current data only
            'historical' - Fetch historical data
            'both' - Fetch both current and historical
        start_date, end_date : str
            For historical mode (format: 'YYYY-MM-DD')
        cities : list
            List of cities (uses all if None)
        """
        
        print("\n" + "="*80)
        print("       INDIA AQI DATA COLLECTION PIPELINE")
        print("       28 STATES + 8 UNION TERRITORIES")
        print("="*80)
        print(f"  Mode: {mode.upper()}")
        print(f"  Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("="*80)
        
        final_data = pd.DataFrame()
        
        if mode in ['current', 'both']:
            current_df = self.fetch_current_data_all_cities()
            if len(current_df) > 0:
                current_df = self.process_data(current_df)
                final_data = pd.concat([final_data, current_df], ignore_index=True)
        
        if mode in ['historical', 'both']:
            if start_date and end_date:
                historical_df = self.fetch_historical_data(
                    start_date, end_date, cities
                )
                if len(historical_df) > 0:
                    historical_df = self.process_data(historical_df)
                    final_data = pd.concat([final_data, historical_df], ignore_index=True)
        
        if len(final_data) > 0:
            final_data = self.format_output(final_data)
        
        print("\n" + "="*80)
        print("  PIPELINE COMPLETE")
        print("="*80)
        print(f"  Total Records: {len(final_data):,}")
        print(f"  Cities: {final_data['City'].nunique() if len(final_data) > 0 else 0}")
        print(f"  States/UTs: {final_data['State'].nunique() if len(final_data) > 0 else 0}")
        print(f"  Columns: {len(final_data.columns) if len(final_data) > 0 else 0}")
        print("="*80)
        
        return final_data


# ============================================
# MAIN EXECUTION
# ============================================

if __name__ == "__main__":
    
    print("""
    â•”â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•—
    â•‘                                                                              â•‘
    â•‘             INDIA AQI DATA COLLECTION - COMPLETE COVERAGE                    â•‘
    â•‘             All 28 States + 8 Union Territories (36 Locations)               â•‘
    â•‘                                                                              â•‘
    â•šâ•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
    """)
    
    # Display all locations
    print("\nðŸ“ Coverage:")
    print("â”€" * 80)
    
    states_count = 0
    uts_count = 0
    
    for city, info in INDIA_CITIES.items():
        state = info.get('state', '')
        if '(UT)' in state or '(NCT)' in state:
            uts_count += 1
        else:
            states_count += 1
    
    print(f"  ðŸ›ï¸  States: {states_count}")
    print(f"  ðŸ™ï¸  Union Territories: {uts_count}")
    print(f"  ðŸ“Š Total Locations: {len(INDIA_CITIES)}")
    print("â”€" * 80)
    
    # Initialize pipeline
    pipeline = IndiaAQIDataPipeline(WAQI_TOKEN)
    
    # ==========================================
    # FETCH CURRENT DATA
    # ==========================================
    
    print("\n" + "="*80)
    print("  FETCHING CURRENT REAL-TIME DATA")
    print("="*80)
    
    current_data = pipeline.run_complete_pipeline(mode='current')
    
    if len(current_data) > 0:
        print("\nðŸ“Š Current Data Sample (First 10 rows):")
        print("â”€" * 80)
        display_cols = ['City', 'State', 'PM2.5', 'PM10', 'Temperature', 
                       'Humidity', 'AQI']
        print(current_data[display_cols].head(10).to_string(index=False))
        
        # Save current data
        filename = f'india_aqi_current_{datetime.now().strftime("%Y%m%d_%H%M")}.csv'
        current_data.to_csv(filename, index=False)
        print(f"\nâœ… Saved to: {filename}")
        
        # Display statistics by state
        print("\nðŸ“ˆ State-wise AQI Statistics:")
        print("â”€" * 80)
        state_stats = current_data.groupby('State')['AQI'].agg(['mean', 'count'])
        state_stats = state_stats.sort_values('mean', ascending=False)
        print(state_stats.head(10).to_string())
    
    # ==========================================
    # DISPLAY SUMMARY
    # ==========================================
    
    print("\n" + "="*80)
    print("  DATA COLLECTION COMPLETE!")
    print("="*80)
    print("\nâœ… Successfully collected data for India AQI prediction")
    print(f"ðŸ“ Output file: {filename if 'filename' in locals() else 'N/A'}")
    print("\nðŸŽ¯ Ready for regression modeling!")
    print("="*80)


    â•”â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•—
    â•‘                                                                              â•‘
    â•‘             INDIA AQI DATA COLLECTION - COMPLETE COVERAGE                    â•‘
    â•‘             All 28 States + 8 Union Territories (36 Locations)               â•‘
    â•‘                                                                              â•‘
    â•šâ•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
    

ðŸ“ Coverage:
â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

TypeError: agg function failed [how->mean,dtype->object]